<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>

# Lane Control

## PID Control

With the state estimation provided by the histogram filter, a PID controller is used to send wheel commands to your Duckiebot.
The PID controller calculates a control signal $u_t$ based on the error $e_t$ between the desired state (reference) and the estimated state:

$$
u_t = K_p e_t + K_i \int_0^t e_{\tau} d \tau + K_d \frac{d e_t}{d_t},
$$

where:

* $K_p$ is the proportional gain, addressing the current error.
* $K_i$ is the integral gain, correcting cumulative past errors.
* $K_d$ is the derivative gain, anticipating future errors based on the rate of change.

For lane following, the control inputs are the reference state $(d_{ref}, \phi_{ref})$ and the control outputs are a constant linear velocity $v_bar$ and a variable angular velocity $\omega$ to correct for deviations.
Using a linearized kinematic model and a constant forward velocity $v$, the control law simplifies to:

$$
u_t = K_d d + K_{\phi} \phi,
$$

where $K_d$ and $K_{\phi}$ are tuned proportional gains for lateral and angular errors, respectively.

## Tuning the PID gains

Tuning the PID gains is one of the most important aspects for the stability and performance of your Duckiebot, as:

* The proportional gain ($K_p$) affects the magnitude of corrections. Too high a value leads to oscillations, while too low a value results in sluggish response.
* The integral gain ($K_i$) addresses steady-state errors but can introduce instability if over tuned.
* The derivative gain ($K_d$) smooths out the response by reducing overshoot but can amplify noise.

These are located in the config for the [lane controller node](../packages/dt-core/packages/lane_control/config/lane_controller_node/). Here are the defaults:

```yaml
v_bar: 0.19
k_d: -2.0
k_theta: -3.0
k_Id: -3.0
k_Iphi: 0.0
theta_thres_max: 0.75 
theta_thres_min: -0.5
d_thres: 0.2615
d_offset: 0.0

integral_bounds:
  d:
    top: 0.3
    bot: -0.3
  phi:
    top: 1.2
    bot: -1.2
```

The are defined as follows: 

- v_bar:  Nominal forward velocity (m/s). This is the cruising speed used when no stop line is detected; velocity is reduced as the robot approaches a stop line.

- k_d: Proportional gain on lateral error d. Negative because a positive lateral deviation (too far left) should produce a negative angular velocity (turn right). Larger magnitude = more aggressive lateral correction.

- k_theta:  Proportional gain on heading error φ. Negative for the same sign convention reason. Typically larger than k_d since heading error is easier to correct quickly.

- k_Id:  Integral gain on lateral error. Accumulates d error over time to eliminate steady-state lateral offset (e.g. a consistent bias from road camber or calibration error).

- k_Iphi: Integral gain on heading error. Currently disabled. Would eliminate steady-state heading bias but can cause oscillation, hence left at zero.

- theta_thres_max / theta_thres_min: Asymmetric clamp on φ error before it enters the control law. Large heading errors (e.g. at intersections) are saturated to prevent extreme angular velocity commands. Asymmetric values allow different sensitivity for left vs. right heading deviations.

- d_thres:  Clamp on lateral error d before it enters the control law (~half a lane width). Prevents the controller from commanding extreme corrections when the robot is far off-center.

- d_offset: Shifts the target lateral position away from lane center. Positive values make the robot drive to the left of center, negative to the right. Useful for avoiding obstacles or adjusting lane position.

- integral_bounds.d: Anti-windup limits for the lateral integral term. The accumulated d error is clamped to this range to prevent integrator windup when the robot is held off-center for a long time.

- integral_bounds.phi: Anti-windup limits for the heading integral term (disabled if k_Iphi = 0).


For more details about PID control please refer to the [Control Learning Experience](https://github.com/duckietown/lx-control).
The velocity and steering values are turned into actuator values using inverse kinematics by [the kinematics node](../packages/dt-core/packages/robots/duckiebot/dagu_car/src/kinematics_node.py). For more details about how this is done please refer to the [Kinematics and Odometry Learning Experience](https://github.com/duckietown/lx-kinematics-odometry).

We now have understood the entire autonomy stack, from the data that comes in through the sensors (camera and encoders) to the actuator commands that are sent to the wheels. One last piece to discuss is how we build a finite state machine that sits on top and manages what the macro-level behaviour of the robot should be. In our case we have two behaviours: `LANE_FOLLOWING`, and `NORMAL_JOYSTICK_CONTROL` that we can toggle between using the `keyboard_control` GUI. This is described in [the next notebook about the finite state machine](./05_finite_state_machine.ipynb).